In [ ]:
%load_ext sql
%sql sqlite:///../store.db
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

# Задача 1.

Вывести 5 самых длинных треков жанра Rock And Roll. Только имя и длина (в секундах).

In [2]:
%%sql

SELECT *
FROM tracks
LIMIT 5

 * sqlite:///../store.db
Done.


TrackId,Name,AlbumId,MediaTypeId,GenreId,Milliseconds,Bytes,UnitPrice
1,For Those About To Rock (We Salute You),1,1,1,343719,11170334,1.99
2,Balls to the Wall,2,2,1,342562,5510424,0.99
3,Fast As a Shark,3,2,1,230619,3990994,2.99
4,Restless and Wild,3,2,1,252051,4331779,1.99
5,Princess of the Dawn,3,2,1,375418,6290521,1.99


In [3]:
%%sql

SELECT *
FROM genres
LIMIT 5

 * sqlite:///../store.db
Done.


GenreId,Name
1,Rock
2,Jazz
3,Metal
4,Alternative & Punk
5,Rock And Roll


In [19]:
%%sql

SELECT
    name,
    milliseconds / 1000 AS time_s,
    "Rock And Roll" AS genre,
    genreId
FROM tracks
WHERE genreId IN (
    SELECT genreId
    FROM genres
    WHERE name = "Rock And Roll"
)
ORDER BY time_s DESC

LIMIT 5

 * sqlite:///../store.db
Done.


Name,time_s,genre,GenreId
Slow Down,163,Rock And Roll,5
Twist And Shout,161,Rock And Roll,5
Money,147,Rock And Roll,5
Roadrunner,143,Rock And Roll,5
Carol,143,Rock And Roll,5


In [32]:
%%sql

SELECT
    tracks_t.name,
    tracks_t.milliseconds / 1000 AS time_s
FROM
    tracks AS tracks_t,
    genres AS genres_t
WHERE
    genres_t.name == "Rock And Roll" AND
    tracks_t.genreId == genres_t.genreId
ORDER BY
    time_s DESC

LIMIT 5

 * sqlite:///../store.db
Done.


Name,time_s
Slow Down,163
Twist And Shout,161
Money,147
Roadrunner,143
Carol,143


In [40]:
%%sql

SELECT
    tracks_t.name,
    tracks_t.milliseconds / 1000 AS time_s
FROM
    tracks AS tracks_t
JOIN
    genres AS genres_t
ON
    genres_t.name == "Rock And Roll" AND
    tracks_t.genreId == genres_t.genreId
ORDER BY
    time_s DESC

LIMIT 5


 * sqlite:///../store.db
Done.


Name,time_s
Slow Down,163
Twist And Shout,161
Money,147
Roadrunner,143
Carol,143


In [25]:
%%sql

SELECT
    tracks_t.name,
    tracks_t.milliseconds / 1000 AS time_s,
    "Rock And Roll" AS genre,
    tracks_t.genreId
FROM
    tracks AS tracks_t
JOIN (
    SELECT genreId
    FROM genres
    WHERE name == "Rock And Roll"
    ) AS genres_t
ON
    tracks_t.genreId == genres_t.genreId
ORDER BY
    time_s DESC

LIMIT 5

 * sqlite:///../store.db
Done.


Name,time_s,genre,GenreId
Slow Down,163,Rock And Roll,5
Twist And Shout,161,Rock And Roll,5
Money,147,Rock And Roll,5
Roadrunner,143,Rock And Roll,5
Carol,143,Rock And Roll,5


# Задача 2.

Вывести названия всех треков и их исполнителей жанра Rock, приобретенных сотрудниками компании Oracle. 

1. Вывести все треки жанра рок

In [80]:
%%sql

SELECT
    tracks_t.name,
    "Rock" AS genre
FROM
    tracks AS tracks_t
INNER JOIN (
    SELECT genreId
    FROM genres
    WHERE name == "Rock"
    ) AS genres_t
ON
    tracks_t.genreId == genres_t.genreId

LIMIT 5

 * sqlite:///../store.db
Done.


Name,genre
For Those About To Rock (We Salute You),Rock
Balls to the Wall,Rock
Fast As a Shark,Rock
Restless and Wild,Rock
Princess of the Dawn,Rock


In [78]:
%%sql

SELECT
    tracks.name,
    "Rock" AS genre
FROM
    tracks
INNER JOIN genres ON
    genres.name == "Rock" AND
    genres.genreId == tracks.genreId

LIMIT 5


 * sqlite:///../store.db
Done.


Name,genre
For Those About To Rock (We Salute You),Rock
Balls to the Wall,Rock
Fast As a Shark,Rock
Restless and Wild,Rock
Princess of the Dawn,Rock


In [82]:
%%sql

SELECT
    tracks.name,
    "Rock" AS genre
FROM
    tracks
INNER JOIN genres ON
    genres.genreId == tracks.genreId
WHERE
    genres.name == "Rock"

LIMIT 5


 * sqlite:///../store.db
Done.


Name,genre
For Those About To Rock (We Salute You),Rock
Balls to the Wall,Rock
Fast As a Shark,Rock
Restless and Wild,Rock
Princess of the Dawn,Rock


In [33]:
%%sql

SELECT
    tracks_t.name,
    "Rock" AS genre
FROM
    tracks AS tracks_t,
    genres AS genres_t
WHERE
    genres_t.name == "Rock" AND
    tracks_t.genreId == genres_t.genreId

LIMIT 5

 * sqlite:///../store.db
Done.


Name,genre
For Those About To Rock (We Salute You),Rock
Balls to the Wall,Rock
Fast As a Shark,Rock
Restless and Wild,Rock
Princess of the Dawn,Rock


## 2. Добавить исполнителей каждого трека

## 2.1. Соеденить альбомы с исполнителями

In [44]:
%%sql

SELECT
    albums.title,
    artists.name
FROM
    albums,
    artists
WHERE
    albums.artistId == artists.artistId

LIMIT 5


 * sqlite:///../store.db
Done.


Title,Name
For Those About To Rock We Salute You,AC/DC
Balls to the Wall,Accept
Restless and Wild,Accept
Let There Be Rock,AC/DC
Big Ones,Aerosmith


Или, что то же самое...

In [83]:
%%sql

SELECT
    albums.title,
    artists.name
FROM
    albums
INNER JOIN
    artists
ON
    albums.artistId == artists.artistId

LIMIT 5


 * sqlite:///../store.db
Done.


Title,Name
For Those About To Rock We Salute You,AC/DC
Balls to the Wall,Accept
Restless and Wild,Accept
Let There Be Rock,AC/DC
Big Ones,Aerosmith


## 2.2. Трек (жанр: рок) + альбом + исполнитель

In [ ]:
%%sql

SELECT
    tracks.name AS "Track Name",
    album_artist_t.title AS "Album Title",
    album_artist_t.artist_name AS "Artist Name"
FROM
    tracks
INNER JOIN genres ON
    genres.name == "Rock" AND tracks.genreId == genres.genreId
INNER JOIN (
    SELECT
        albums.title AS title,
        artists.name AS artist_name,
        albums.albumId
    FROM albums
    INNER JOIN artists ON albums.artistId == artists.artistId
) AS album_artist_t
ON
    tracks.albumId == album_artist_t.albumId

LIMIT 7

 * sqlite:///../store.db
Done.


Track Name,Album Title,Artist Name
For Those About To Rock (We Salute You),For Those About To Rock We Salute You,AC/DC
Balls to the Wall,Balls to the Wall,Accept
Fast As a Shark,Restless and Wild,Accept
Restless and Wild,Restless and Wild,Accept
Princess of the Dawn,Restless and Wild,Accept
Put The Finger On You,For Those About To Rock We Salute You,AC/DC
Let's Get It Up,For Those About To Rock We Salute You,AC/DC


Или, что то же самое...

In [85]:
%%sql

SELECT
    tracks_rock.name AS "Track Name",
    album_artist_t.title AS "Album Title",
    album_artist_t.artist_name AS "Artist Name"
FROM (
    SELECT *
    FROM tracks, genres
    WHERE
        genres.name == "Rock" AND
        tracks.genreId == genres.genreId
) AS tracks_rock
INNER JOIN (
    SELECT
        albums.title AS title,
        artists.name AS artist_name,
        albums.albumId
    FROM albums, artists
    WHERE albums.artistId == artists.artistId
) AS album_artist_t
ON
    tracks_rock.albumId == album_artist_t.albumId

LIMIT 7


 * sqlite:///../store.db
Done.


Track Name,Album Title,Artist Name
For Those About To Rock (We Salute You),For Those About To Rock We Salute You,AC/DC
Balls to the Wall,Balls to the Wall,Accept
Fast As a Shark,Restless and Wild,Accept
Restless and Wild,Restless and Wild,Accept
Princess of the Dawn,Restless and Wild,Accept
Put The Finger On You,For Those About To Rock We Salute You,AC/DC
Let's Get It Up,For Those About To Rock We Salute You,AC/DC


## 3. Сотрудники Oracle

In [92]:
%%sql

SELECT 
    firstname || ' ' || lastname AS "Name",
    company
FROM customers
WHERE company == "Oracle"

LIMIT 10

 * sqlite:///../store.db
Done.


Name,Company
John Gordon,Oracle
Frank Ralston,Oracle


## 4. Объединяем все через invoice_items и invoices

## 4.1. trackId + invoiceId + customerId

In [107]:
%%sql

SELECT
    oracle.name AS "Customer Name",
    items.invoiceId
FROM invoice_items AS items
INNER JOIN invoices ON
    items.invoiceId == invoices.invoiceId
INNER JOIN (
    SELECT
        firstname || ' ' || lastname AS name,
        company,
        customerId
    FROM customers
    WHERE company == "Oracle"
) AS oracle ON
    invoices.customerId == oracle.customerId

LIMIT 5

 * sqlite:///../store.db
Done.


Customer Name,InvoiceId
John Gordon,5
John Gordon,5
John Gordon,5
John Gordon,5
John Gordon,5


In [105]:
%%sql

SELECT
    custs.firstname || ' ' || custs.lastname AS "Customer Name",
    items.invoiceId
FROM invoice_items AS items
INNER JOIN invoices ON
    items.invoiceId == invoices.invoiceId
INNER JOIN customers AS custs ON
    invoices.customerId == custs.customerId
WHERE custs.company == "Oracle"

LIMIT 5

 * sqlite:///../store.db
Done.


Customer Name,InvoiceId
John Gordon,5
John Gordon,5
John Gordon,5
John Gordon,5
John Gordon,5


## 4.2. Объединение треков (жанр: рок) и заказов (из Oracle)

v1

In [114]:
%%sql

SELECT
    invoices_oracle.cust_name AS "C. Name",
    tracks.name AS "Track Name",
    album_artist_t.artist_name AS "Artist Name"
FROM
    tracks
INNER JOIN genres ON
    genres.name == "Rock" AND tracks.genreId == genres.genreId
INNER JOIN (
    SELECT
        albums.title AS title,
        artists.name AS artist_name,
        albums.albumId
    FROM albums
    INNER JOIN artists ON albums.artistId == artists.artistId
) AS album_artist_t
ON
    tracks.albumId == album_artist_t.albumId
INNER JOIN (
    SELECT
        oracle.name AS cust_name,
        items.trackId
    FROM invoice_items AS items
    INNER JOIN invoices ON
        items.invoiceId == invoices.invoiceId
    INNER JOIN (
        SELECT
            firstname || ' ' || lastname AS name,
            company,
            customerId
        FROM customers
        WHERE company == "Oracle"
    ) AS oracle ON
        invoices.customerId == oracle.customerId
) AS invoices_oracle
ON
    tracks.trackId == invoices_oracle.trackId


LIMIT 7

 * sqlite:///../store.db
Done.


C. Name,Track Name,Artist Name
Frank Ralston,Sunday Bloody Sunday,U2
Frank Ralston,New Year's Day,U2
Frank Ralston,That's The Way,Led Zeppelin
Frank Ralston,Ten Years Gone,Led Zeppelin
Frank Ralston,Achilles Last Stand,Led Zeppelin
Frank Ralston,Tea For One,Led Zeppelin
Frank Ralston,No Quarter,Led Zeppelin


v2

Вместо tracks используется подзапрос tracks_rock

In [113]:
%%sql

SELECT
    items_oracle.customer AS "Customer",
    tracks_rock.name AS "Track",
    artist_album.artist_name AS "Artist"
FROM (
    SELECT *
    FROM tracks
    INNER JOIN genres ON
        genres.name == "Rock" AND tracks.genreId == genres.genreId
) AS tracks_rock
INNER JOIN (
    SELECT
        artists.name AS artist_name,
        albums.albumId
    FROM albums
    INNER JOIN artists ON
        albums.artistId == artists.artistId
) AS artist_album
ON
    tracks_rock.albumId == artist_album.albumId
INNER JOIN (
    SELECT
        items.trackId,
        oracle.name AS customer
    FROM invoice_items AS items
    INNER JOIN invoices ON
        items.invoiceId == invoices.invoiceId
    INNER JOIN (
        SELECT
            firstname || ' ' || lastname AS name,
            company,
            customerId
        FROM customers
        WHERE company == "Oracle"
    ) AS oracle ON
        invoices.customerId == oracle.customerId
) AS items_oracle
ON
    tracks_rock.trackId == items_oracle.trackId

LIMIT 7

 * sqlite:///../store.db
Done.


Customer,Track,Artist
Frank Ralston,Sunday Bloody Sunday,U2
Frank Ralston,New Year's Day,U2
Frank Ralston,That's The Way,Led Zeppelin
Frank Ralston,Ten Years Gone,Led Zeppelin
Frank Ralston,Achilles Last Stand,Led Zeppelin
Frank Ralston,Tea For One,Led Zeppelin
Frank Ralston,No Quarter,Led Zeppelin


v3

Оптимизация join'ов заказчиков из Oracle

In [140]:
%%sql

SELECT
    items_oracle.customer AS "Customer",
    tracks_rock.name AS "Track",
    artist_album.artist_name AS "Artist"
FROM (
    SELECT *
    FROM tracks
    INNER JOIN genres ON
        genres.name == "Rock" AND tracks.genreId == genres.genreId
) AS tracks_rock
INNER JOIN (
    SELECT
        artists.name AS artist_name,
        albums.albumId
    FROM albums
    INNER JOIN artists ON
        albums.artistId == artists.artistId
) AS artist_album
ON
    tracks_rock.albumId == artist_album.albumId
INNER JOIN (
    SELECT
        items.trackId,
        oracle.customer
    FROM (
        SELECT
            firstname || ' ' || lastname AS customer,
            customerId
        FROM customers
        WHERE company == "Oracle"
    ) AS oracle
    INNER JOIN invoices ON
        oracle.customerId == invoices.customerId
    INNER JOIN invoice_items AS items ON
        invoices.invoiceId == items.invoiceId
) AS items_oracle
ON
    tracks_rock.trackId == items_oracle.trackId

LIMIT 7


 * sqlite:///../store.db
Done.


Customer,Track,Artist
Frank Ralston,Sunday Bloody Sunday,U2
Frank Ralston,New Year's Day,U2
Frank Ralston,That's The Way,Led Zeppelin
Frank Ralston,Ten Years Gone,Led Zeppelin
Frank Ralston,Achilles Last Stand,Led Zeppelin
Frank Ralston,Tea For One,Led Zeppelin
Frank Ralston,No Quarter,Led Zeppelin


# Задача 3.

Для каждой компании вывести общее количество купленных джазовых композиций (если 0, то не выводить)

In [151]:
%%sql

SELECT
    comp AS "Company",
    SUM(comps.quantity) AS "Quantity"
FROM (
    SELECT
        customers.company AS comp,
        items.trackId,
        items.quantity
    FROM customers
    INNER JOIN invoices ON
        customers.customerId == invoices.customerId
    INNER JOIN invoice_items AS items ON
        invoices.invoiceId == items.invoiceId
    WHERE comp IS NOT NULL
) AS comps
INNER JOIN (
    SELECT trackId
    FROM tracks
    INNER JOIN (
        SELECT genreId
        FROM genres
        WHERE genres.name == "Jazz"
    ) AS genres_jazz ON
            tracks.genreId == genres_jazz.genreId
) AS tracks_jazz
ON
    comps.trackId == tracks_jazz.trackId
GROUP BY
    comp
HAVING
    SUM(quantity) > 0
ORDER BY
    Quantity DESC

LIMIT 15

 * sqlite:///../store.db
Done.


Company,Quantity
Microsoft Corporation,19
Google Inc.,16
Oracle,3
JetBrains s.r.o.,3
Apple Inc.,3
Telus,2
SAP,2
BMW,1


# Задача 4.

Вывести топ 3 плейлиста по продолжительности

In [158]:
%%sql

SELECT
    playlists.name AS "Playlist",
    SUM(tracks.milliseconds) / 1000 AS "Time, s"
FROM
    playlist_track
INNER JOIN tracks ON
    playlist_track.trackId == tracks.trackId
INNER JOIN playlists ON
    playlist_track.playlistId == playlists.playlistId
GROUP BY
    playlists.name
ORDER BY
    SUM(tracks.milliseconds) / 1000 DESC

LIMIT 3

 * sqlite:///../store.db
Done.


Playlist,"Time, s"
From That TV Show,498863
Hidden Gems,498725
90’s Music,37981


# Задача 5.

Для каждого набора (жанр, тип медиа) вывести количество треков в них, причем вывести только те наборы, в которых все треки стоят больше 1$ и для которых в наборе есть хотя бы один трек

In [192]:
%%sql

SELECT
    genres.name || ' [' || mtypes.name || ']' AS genre_mtype,
    COUNT(t.trackId) AS tracks_quantity
FROM (
    SELECT trackId, genreId, mediaTypeId
    FROM tracks
    WHERE unitprice > 1.0
) AS t
INNER JOIN genres ON
    t.genreId == genres.genreId
INNER JOIN media_types AS mtypes ON
    t.mediaTypeId == mtypes.mediaTypeId
GROUP BY
    genre_mtype
HAVING
    tracks_quantity > 0
LIMIT 15 OFFSET 23

 * sqlite:///../store.db
Done.


genre_mtype,tracks_quantity
Metal [protected AAC audio file],4
Pop [FLAC],6
R&B/Soul [FLAC],18
R&B/Soul [protected AAC audio file],5
Reggae [FLAC],5
Reggae [WMA audio],7
Rock And Roll [FLAC],5
Rock [AAC audio file],40
Rock [FLAC],188
Rock [MPEG audio file],24
